# 1 — Data Exploration

This notebook downloads, loads, and visualises VQA v2.0 data to verify the pipeline works.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import json
import matplotlib.pyplot as plt
from collections import Counter
from pathlib import Path
from PIL import Image

DATA_DIR = Path("../data")

## 1.1 — Download the dataset

Run this once.  Add `--download-images` to also fetch MS-COCO images (~19 GB).

In [ ]:
# !python -m data.download_data --data-dir ../data
# !python -m data.download_data --data-dir ../data --download-images

## 1.2 — Inspect questions and annotations

In [ ]:
with open(DATA_DIR / "questions" / "v2_OpenEnded_mscoco_train2014_questions.json") as f:
    questions = json.load(f)["questions"]
with open(DATA_DIR / "answers" / "v2_mscoco_train2014_annotations.json") as f:
    annotations = json.load(f)["annotations"]

print(f"Questions: {len(questions):,}")
print(f"Annotations: {len(annotations):,}")
print("\nSample question:", questions[0])
print("\nSample annotation:", annotations[0])

## 1.3 — Answer distribution

In [ ]:
answer_counts = Counter(ann["multiple_choice_answer"] for ann in annotations)
top_20 = answer_counts.most_common(20)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh([a for a, _ in top_20][::-1], [c for _, c in top_20][::-1])
ax.set_xlabel("Count")
ax.set_title("Top 20 Answers")
plt.tight_layout()
plt.show()

# Coverage analysis
total = len(annotations)
for k in [100, 500, 1000, 2000]:
    covered = sum(c for _, c in answer_counts.most_common(k))
    print(f"Top {k:>5d} answers cover {covered/total*100:.1f}% of samples")

## 1.4 — Visualise sample image-question-answer triplets

In [ ]:
ann_by_qid = {ann["question_id"]: ann for ann in annotations}
img_dir = DATA_DIR / "images" / "train2014"

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, q in zip(axes.flat, questions[:6]):
    img_id = q["image_id"]
    img_path = img_dir / f"COCO_train2014_{img_id:012d}.jpg"
    answer = ann_by_qid[q["question_id"]]["multiple_choice_answer"]
    if img_path.exists():
        ax.imshow(Image.open(img_path))
    ax.set_title(f"Q: {q['question']}\nA: {answer}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 1.5 — Test the VQADataset class

In [ ]:
from data.preprocess import VQADataset, build_answer_vocab, get_image_transform

answer_to_idx, idx_to_answer = build_answer_vocab(
    DATA_DIR / "answers" / "v2_mscoco_train2014_annotations.json",
    top_k=1000,
)
print(f"Answer vocab size: {len(answer_to_idx)}")

ds = VQADataset(
    questions_file=DATA_DIR / "questions" / "v2_OpenEnded_mscoco_train2014_questions.json",
    annotations_file=DATA_DIR / "answers" / "v2_mscoco_train2014_annotations.json",
    image_dir=DATA_DIR / "images" / "train2014",
    answer_to_idx=answer_to_idx,
    transform=get_image_transform("val"),
    max_samples=100,
)
print(f"Dataset size: {len(ds)}")

image, input_ids, attention_mask, answer_idx = ds[0]
print(f"Image shape: {image.shape}")
print(f"Input IDs shape: {input_ids.shape}")
print(f"Answer: {idx_to_answer[answer_idx.item()]}")